# scSVC reconstructs whole-genome profiles of Xenium datasets and defines fine-grained subtypes of T cells

In [ ]:
output_dir = "../../output/sc_SVC_case/P2CRC_Xenium"
select_ct = "T"


In [ ]:
import os
import scanpy as sc

from revise.application.sc_svc import ScSVCAnalysis

svc_save_dir = f"{output_dir}/{select_ct}"
sc_svc_expr = sc.read_h5ad(f"{svc_save_dir}/sc_SVC_expr.h5ad")
sc_svc_spatial = sc.read_h5ad(f"{svc_save_dir}/sc_SVC_spatial.h5ad")

sc_svc_analysis = ScSVCAnalysis(sc_svc_spatial, sc_svc_expr, 
                            "SVC_cluster")

In [ ]:
cm_df = sc_svc_analysis.get_cm_df("Level2")
cm_df

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors 

size = None
cmap = plt.cm.get_cmap('tab20', lut=10)
palette = [mcolors.to_hex(cmap(i)) for i in range(cmap.N)]

sc.pl.scatter(sc_svc_analysis.sc_SVC_adata_spatial, x="x", y="y",
              color='Level2',
              size=size)
desired_order = ['1', '2', '3', '5', '4', '6', '0']  # for better visualization
sc_svc_analysis.sc_SVC_adata_spatial.obs['SVC_cluster'] = (
    sc_svc_analysis.sc_SVC_adata_spatial.obs['SVC_cluster'].cat.reorder_categories(desired_order, ordered=True)
)
sc.pl.scatter(sc_svc_analysis.sc_SVC_adata_spatial, x="x", y="y",
              color='SVC_cluster',
              size=size)

In [ ]:
import matplotlib.pyplot as plt
def plot_sc_SVC(adata, color, title = None, file_name = None):

    plt.figure(figsize=(10, 8*len(color)))
    sc.pl.scatter(
        adata, x="x", y="y",
        color = color,
        title=title, show = False,
            )
    plt.savefig(file_name, dpi = 300)
    plt.close()

sc_SVC_file_name = f"{output_dir}/sc_SVC.png"
plot_sc_SVC(sc_svc_analysis.sc_SVC_adata_spatial, color='SVC_cluster',
            file_name=sc_SVC_file_name
            )

sc_SVC_file_name = f"{output_dir}/compare.png"
plot_sc_SVC(sc_svc_analysis.sc_SVC_adata_spatial, color=['Level2','SVC_cluster'],
            title=["Expert anno", "sc_SVC"],
            file_name=sc_SVC_file_name
            )

## bioinfo analysis

In [ ]:
cluster_nums = ['1','3','5']
fc_threshold = 1
pathway_num = 20
gene_num = 60
geneset_file = ["MSigDB_Hallmark_2020"]
pathway_file_name = f"{output_dir}/pathway_{fc_threshold}_{pathway_num}.txt"
all_pathway = sc_svc_analysis.get_pathway_conclusion(
    cluster_nums, fc_threshold=fc_threshold, pathway_num=pathway_num, gene_num=gene_num, geneset_file=geneset_file, normalize=True)
all_pathway.to_csv(pathway_file_name)
all_pathway

In [ ]:
cluster_nums = ['1', '3', '5']

degs = sc_svc_analysis.get_svc_degs(cluster_nums, fc_threshold=1)
marker_dict = (
    degs.groupby('group')['gene']
    .apply(lambda x: x.head(5).tolist())
    .to_dict()
)
sc_svc_analysis.get_dot_plot(cluster_nums, marker_dict)

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.useafm'] = False
plt.figure(figsize=(8, 6))

marker_dict = {
    '1': ['MKI67', 'TYMS', 'RRM2'],
    '3': ['CCL5', 'IL7R', 'GZMA'],
    '5': ['FOXP3', 'CTLA4', 'MAF']
    }

sc_svc_analysis.get_dot_plot(cluster_nums, marker_dict, normalize=True)
plt.savefig(f"{output_dir}/sc_SVC_dotplot.pdf", dpi=300, bbox_inches='tight')
plt.close()

### sc_SVC

In [ ]:
# tumor_cluster_num = '1' # Proliferation
tumor_cluster_num = '5' # Regulation

# tumor_cluster = '0' # Median
normal_cluster = '3' # normal

if tumor_cluster_num == '1':
    tumor_cluster = 'Proliferation'
elif tumor_cluster_num == '5':
    tumor_cluster = 'Regulation'

In [ ]:
import pandas as pd
# for statical numbers of degs and pathways 
pvals_adj = 0.01
logfoldchanges = 1
num_df = pd.DataFrame(0, index=['sp_SC', 'raw_Xenium'], columns=['DEG', 'Pathway'])

cluster_nums = [normal_cluster, tumor_cluster_num]
replace_cols = {normal_cluster: 'Normal-infiltrated', tumor_cluster_num: 'Tumor-infiltrated'}
degs = sc_svc_analysis.get_volcano_plot(cluster_nums, target_group="Tumor-infiltrated", replace_cols=replace_cols, fc_threshold=None, log_fold_changes=10, logfc_threshold=1, padj_threshold=1e-6, top_k=10)

deg_num = ((degs['pvals_adj'] < pvals_adj) & (degs['logfoldchanges'].abs() > logfoldchanges)).sum()
print( f"sc_SVC {tumor_cluster} degs num: ", deg_num )
num_df.loc['sp_SC', 'DEG'] = deg_num

plt.savefig(f"{output_dir}/sc_SVC/{tumor_cluster}_volcano.pdf", dpi=300, bbox_inches='tight')

In [ ]:
# for enrichment analysis
from revise.tools.bio import get_enrichment, pathway_barplot, pathway_network_plot

degs = degs[degs['logfoldchanges'] > 0]
degs.reset_index(drop = True, inplace = True)
deg_genes = degs["gene"][:60].tolist()
pathway = get_enrichment(deg_genes, geneset_file)
pathway.to_csv(f"{output_dir}/sc_SVC/{tumor_cluster}_pathway.csv", index=False)

pathway_barplot(pathway)
pathway_network_plot(pathway, 
                     top_term = 6, 
                     save_file_name = f'{output_dir}/sc_SVC/{tumor_cluster}_network.pdf')

### raw

In [ ]:
from revise.tools.bio import get_degs
os.makedirs(f"{output_dir}/raw_Xenium", exist_ok=True)
sc_SVC_adata = sc_svc_analysis.sc_SVC_adata_spatial.copy()
raw_select_adata = sc_SVC_adata[sc_SVC_adata.obs['SVC_cluster'].isin([normal_cluster, tumor_cluster_num])]
raw_select_adata.obs['SVC_cluster'].replace({normal_cluster: 'Normal-infiltrated', tumor_cluster_num: 'Tumor-infiltrated'}, inplace=True)
raw_deg_df = get_degs(raw_select_adata, groupby='SVC_cluster', method='t-test', fc_threshold=None)
raw_deg_df = raw_deg_df[raw_deg_df['group'] == "Tumor-infiltrated"]
raw_deg_df.reset_index(drop = True, inplace = True)
raw_deg_df.to_csv(f"{output_dir}/raw_Xenium/{tumor_cluster}_degs.csv", index=False)

from revise.tools.bio import plot_volcano
plot_volcano(raw_deg_df, logfc_threshold=1, padj_threshold=1e-6, 
                 top_k=10, save_file_name = f"{output_dir}/raw_Xenium/{tumor_cluster}_volcano.pdf")


In [ ]:
# for enrichment analysis
from revise.tools.bio import get_enrichment
raw_deg_df = raw_deg_df[raw_deg_df['logfoldchanges'] > 0] # postive for up-regulated pathway
raw_deg_df.reset_index(drop = True, inplace = True)
deg_genes = raw_deg_df["gene"][:60].tolist()
pathway = get_enrichment(deg_genes, geneset_file)
pathway.to_csv(f"{output_dir}/raw_Xenium/{tumor_cluster}_pathway.csv", index=False)

raw_pathway_num = (pathway['Adjusted P-value'] < pvals_adj).sum()
print( f"Raw Xenium {tumor_cluster} pathway num: ", raw_pathway_num )
num_df.loc['raw_Xenium', 'Pathway'] = raw_pathway_num
num_df.to_csv(f"{output_dir}/sc_SVC/{tumor_cluster}_num.csv", index=False)

pathway_barplot(pathway)
pathway_network_plot(pathway, 
                     top_term = 6, 
                     save_file_name = f'{output_dir}/raw_Xenium/{tumor_cluster}_network.pdf')

In [ ]:
# for enrichment analysis
from sc_SVC_utils import get_enrichment
raw_deg_df = raw_deg_df[raw_deg_df['logfoldchanges'] > 0] # postive for up-regulated pathway
raw_deg_df.reset_index(drop = True, inplace = True)
deg_genes = raw_deg_df["gene"][:60].tolist()
pathway = get_enrichment(deg_genes, geneset_file)
pathway.to_csv(f"{output_dir}/raw_Xenium/{tumor_cluster}_pathway.csv", index=False)
raw_pathway_num = (pathway['Adjusted P-value'] < pvals_adj).sum()
print( f"Raw Xenium {tumor_cluster} pathway num: ", raw_pathway_num )
num_df.loc['raw_Xenium', 'Pathway'] = raw_pathway_num
num_df.to_csv(f"{output_dir}/sc_SVC/{tumor_cluster}_num.csv", index=False)

pathway_barplot(pathway)
pathway_network_plot(pathway, 
                     top_term = 6, 
                     save_file_name = f'{output_dir}/raw_Xenium/{tumor_cluster}_network.pdf')